# 03 — Implement crop

Select a spectral band through STAC metadata and verify the cropped grid promised by the contract.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/03-implement-crop/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/03-implement-crop/lesson.md) · [Exercise](../modules/03-implement-crop/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/03-implement-crop"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Prepare and crop a local scene

Predict the width and height of an AOI spanning `1,1,7,7` with one-degree pixels. `--band green` resolves an EO common band name, rather than guessing an asset filename.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
from scripts.fixtures import create_fixture
create_fixture(Path('source'))
print('Created an 8 x 8 STAC scene with green and NIR assets under source/')
PYTHON

waterbodies crop --input-item source --aoi "1,1,7,7" --epsg EPSG:4326 --band green
)


## Inspect the actual output

A valid file needs the correct bounds, CRS, transform and COG layout. Inspect the generated output rather than relying only on a successful process exit.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
import rasterio
with rasterio.open('crop_green.tif') as raster:
    print('Shape:', raster.shape, 'Bounds:', raster.bounds)
    print('CRS:', raster.crs, 'Transform:', raster.transform)
    print('Layout:', raster.tags(ns='IMAGE_STRUCTURE'))
    assert raster.shape == (6, 6)
    assert tuple(raster.bounds) == (1, 1, 7, 7)
    assert raster.tags(ns='IMAGE_STRUCTURE')['LAYOUT'] == 'COG'
PYTHON
)


## Test a disjoint AOI

Predict the failure for a box completely outside the scene. The following command must be rejected; returning an unrelated crop would violate the caller's intent.


In [ ]:
(
set -euo pipefail
if waterbodies crop --input-item source --aoi "100,100,101,101" --epsg EPSG:4326 --band green > disjoint.log 2>&1; then
    printf 'Unexpected success for a disjoint AOI.\n' >&2
    exit 1
fi
cat disjoint.log
python -m pytest "$WORKSPACE/tests/tools/test_processing.py" -k crop
)


## Experiment

Change the first AOI to `2,2,7,7` and predict the new grid. Adjust your inspection assertions to the intended bounds, then restore the default AOI. Read `crop_impl.py` in the editor and trace where coordinate transformation and intersection checks occur.


## Record your findings

**Expected outcome:** The default crop is a 6×6 COG spanning 1,1,7,7; disjoint AOIs fail and crop tests pass.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/03-implement-crop/solution.md) after the activity. Return to [the lab index](README.md).
